# Dataset Merge Walkthrough

This generated notebook is the recipe companion for
`examples/api_dataset_merge_demo.py`.

Combine raw and processed datasets, handle mnemonic collisions safely, and inspect the merge history captured in dataset provenance.

What you will practice in this walkthrough:

- Separate raw acquisition data from derived or QC datasets before merging them.
- Choose an explicit collision policy instead of silently overwriting matching mnemonics.
- Use merge-history provenance to understand how the final working dataset was assembled.

Prerequisites:

- `pip install "wellplot[notebook]"`
- run the notebook from a checkout of this repository so the `examples/` files and sample data are available

Runtime model:

- import `wellplot` from the active installed environment
- use the repository checkout for the example files, helper modules, and sample data


In [ ]:
import sys
from pathlib import Path

try:
    import wellplot
except ImportError as exc:
    raise RuntimeError(
        "Install the published 'wellplot' package in the active "
        "environment before running this notebook."
    ) from exc

# Walk upward from the current working directory until we find the
# repository checkout that holds the example sources and sample data.
cwd = Path.cwd().resolve()
REPO_ROOT = next((path for path in (cwd, *cwd.parents) if (path / "examples").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError(
        "Run this notebook from a checkout of the wellplot repository "
        "so the example files and sample data are available."
    )

EXAMPLES_DIR = REPO_ROOT / "examples"
WORKSPACE_DIR = REPO_ROOT / "workspace"
WORKSPACE_RENDERS = WORKSPACE_DIR / "renders"
WORKSPACE_RENDERS.mkdir(parents=True, exist_ok=True)

examples_path = str(EXAMPLES_DIR)
if examples_path not in sys.path:
    sys.path.insert(0, examples_path)

print("wellplot version:", wellplot.__version__)
print("Examples root:", EXAMPLES_DIR)
print("Render output:", WORKSPACE_RENDERS)

In [ ]:
# Display the source directly in the notebook so the recipe is easy to
# read and copy from without opening another file.
from IPython.display import Code, display

source_path = EXAMPLES_DIR / "api_dataset_merge_demo.py"
display(Code(source_path.read_text(), language="python"))

In [ ]:
# Import the example module and the merge builder helper used in the script.
import api_dataset_merge_demo as demo
from wellplot import DatasetBuilder

# Build the raw and processed inputs separately so you can inspect what is
# about to be merged and decide how to handle name collisions.
raw = demo.build_raw_dataset()
processed = demo.build_processed_dataset()
print("Raw channels:", sorted(raw.channels))
print("Processed channels:", sorted(processed.channels))

# Merge with collision='rename' so the derived GR channel remains available
# without replacing the raw GR curve.
merged = (
    DatasetBuilder(name="merged")
    .merge(raw, merge_well_metadata=True, merge_provenance=True)
    .merge(
        processed,
        collision="rename",
        rename_template="{mnemonic}_{dataset}",
    )
    .build()
)

print("Merged channels:", sorted(merged.channels))
print("Merge history:", merged.provenance["merge_history"])
print("Renamed channel metadata:", merged.get_channel("GR_qc").metadata)

## Adapt Dataset Merge Walkthrough Safely

- Use `collision='rename'` when both the raw and processed versions of a mnemonic need to stay visible.
- Use provenance merges when downstream users need to understand where each channel came from.
- Promote the merged dataset to your layout layer only after the final channel names are stable.
